In [1]:
input_geotiff = 'los_angeles_fires_2025/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250303T140056Z_20251016T224900Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250303T140056Z_20251016T224900Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'
rgb_tiff = 'rgb.tif'

In [2]:
import rasterio
import numpy as np

def indexed_to_rgb_robust(input_path, output_path):
    with rasterio.open(input_path) as src:
        colormap = src.colormap(1)
        if colormap is None:
            return

        indexed_data = src.read(1)

        max_index = max(colormap.keys())
        colormap_array = np.zeros((max_index + 1, 4), dtype=np.uint8)

        for index, rgba in colormap.items():
            colormap_array[index] = rgba

        rgb_rgba_data = colormap_array[indexed_data]

        rgb_data = rgb_rgba_data[:, :, :3].transpose(2, 0, 1)

        profile = src.profile
        profile.update(
            dtype=rasterio.uint8, 
            count=3,           
            nodata=None        
        )

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(rgb_data, indexes=[1, 2, 3])
            print(f"Successfully converted and saved RGB file to {output_path}")

In [3]:
indexed_to_rgb_robust(input_geotiff, rgb_tiff)

Successfully converted and saved RGB file to rgb.tif


In [4]:
!rio pmtiles {rgb_tiff} los_angeles_2025.pmtiles --format PNG --resampling nearest --zoom-levels 5..15

100%|████████████████████████████████████| 16102/16102 [00:27<00:00, 581.82it/s]


In [5]:
import geopandas as gpd

In [6]:
df = gpd.read_parquet('/Users/cmarshak/bekaert-team/dist-s1-events/db/event_perimeters/los_angeles_fires_2025.parquet')


In [7]:
df.columns.tolist()

['OBJECTID',
 'poly_SourceOID',
 'poly_IncidentName',
 'poly_FeatureCategory',
 'poly_MapMethod',
 'poly_GISAcres',
 'poly_DeleteThis',
 'poly_FeatureAccess',
 'poly_FeatureStatus',
 'poly_IsVisible',
 'poly_CreateDate',
 'poly_DateCurrent',
 'poly_PolygonDateTime',
 'poly_IRWINID',
 'poly_FORID',
 'poly_Acres_AutoCalc',
 'poly_SourceGlobalID',
 'poly_Source',
 'attr_SourceOID',
 'attr_ABCDMisc',
 'attr_ADSPermissionState',
 'attr_CalculatedAcres',
 'attr_ContainmentDateTime',
 'attr_ControlDateTime',
 'attr_CreatedBySystem',
 'attr_IncidentSize',
 'attr_DiscoveryAcres',
 'attr_DispatchCenterID',
 'attr_EstimatedCostToDate',
 'attr_FinalAcres',
 'attr_FFReportApprovedByTitle',
 'attr_FFReportApprovedByUnit',
 'attr_FFReportApprovedDate',
 'attr_FireBehaviorGeneral',
 'attr_FireBehaviorGeneral1',
 'attr_FireBehaviorGeneral2',
 'attr_FireBehaviorGeneral3',
 'attr_FireCause',
 'attr_FireCauseGeneral',
 'attr_FireCauseSpecific',
 'attr_FireCode',
 'attr_FireDepartmentID',
 'attr_FireDiscov

In [8]:
df_f = df[['poly_IncidentName', 'start_time', 'geometry']].copy()
df_f = df_f.rename(columns={'poly_IncidentName': 'fire_name'})
df_f.fire_name = df_f.fire_name.str.lower()
df_f.head()

,fire_name,start_time,geometry
0,palisades,2025-01-07 19:23:56,"MULTIPOLYGON (((-118.56093 34.04408, -118.5609..."
1,eaton,2025-01-08 02:22:48,"MULTIPOLYGON (((-118.08301 34.23769, -118.0830..."
2,hurst,2025-01-08 06:53:19,"POLYGON ((-118.47458 34.34255, -118.47434 34.3..."
3,freddy,2025-01-08 20:42:48,"POLYGON ((-118.94196 34.04779, -118.94196 34.0..."
4,laguna,2025-01-23 16:40:22,"POLYGON ((-119.05553 34.16243, -119.05518 34.1..."


In [9]:
df_f.to_file('los_angeles_fires.geojson', driver='GeoJSON')